[Reference](https://medium.com/@shubhamnagar234/tool-calling-in-langchain-how-llms-use-tools-to-take-action-part-17-361a4ffdce5d$0)

# Tool Binding
```
Tool
 ├── Name
 ├── Description
 ├── Input
 └── Input Schema
        │
        ▼
   Tool Binding
        │
        ▼
       LLM
```

In [1]:
# If using ChatOpenAI
# from langchain_openai import ChatOpenAI

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage


# Create a custom tool
@tool
def multiply(a: int, b: int) -> int:
    """Given 2 numbers a and b this tool returns their product"""
    return a * b


# Test the tool directly
print(multiply.invoke({"a": 3, "b": 4}))


# Tool binding

# If using ChatOpenAI
# llm = ChatOpenAI()

# If using ChatGoogleGenerativeAI
# Initialize ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI()


# Test the LLM
print(llm.invoke("hi"))


# Bind the tool to the LLM
llm_with_tools = llm.bind_tools([multiply])


# Invoke the LLM with the bound tool
print(llm_with_tools.invoke("Hi how are you"))

# Tool Calling
```
LLM
 │
 │ decides which tool to use
 ▼
Tool Call
 │
 ├── tool: multiply
 └── args: a=8, b=7
 │
 ▼
Application executes the tool
 │
 ▼
56
```

In [2]:
# If using ChatOpenAI
# from langchain_openai import ChatOpenAI

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage


# Create a custom tool
@tool
def multiply(a: int, b: int) -> int:
    """Given 2 numbers a and b this tool returns their product"""
    return a * b


# Test the tool directly
print(multiply.invoke({"a": 3, "b": 4}))


# Initialize the LLM

# If using ChatOpenAI
# llm = ChatOpenAI()

# If using ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI()


# Bind the tool to the LLM
llm_with_tools = llm.bind_tools([multiply])


# Tool calling
print(llm_with_tools.invoke("Hi how are you").tool_calls[0])

# Tool Execution

In [3]:
# If using ChatOpenAI
# # from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

# tool create


@tool
def multiply(a: int, b: int) -> int:
    """Given 2 numbers a and b this tool returns their product"""
    return a * b


print(multiply.invoke({"a": 3, "b": 4}))

# tool binding
# If using ChatOpenAI
# llm = ChatOpenAI()

# If using GoogleGenerativeAIEmbeddings
# Initialize ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI()

# print(llm.invoke("hi"))

llm_with_tools = llm.bind_tools([multiply])

# tool calling
print(llm_with_tools.invoke("Hi how are you").tool_calls[0])

query = HumanMessage("can you multiply 3 with 1000")

messages = [query]
# print(messages)

result = llm_with_tools.invoke(messages)

messages.append(result)
print(messages)

tool_result = multiply.invoke(result.tool_calls[0])
print(tool_result)

messages.append(tool_result)
print(messages)

print(llm_with_tools.invoke(messages).content)

# Currency Conversion Tool

In [4]:
# If using ChatOpenAI
# # from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated
from langchain.agents import initialize_agent, AgentType


@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor between a given base currency and a target currency
    """
    url = f"https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}"

    response = requests.get(url)

    return response.json()


@tool
def convert(
    base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]
) -> float:
    """
    given a currency conversion rate this function calculates the target currency value from a given base currency value
    """

    return base_currency_value * conversion_rate


# print(convert.args)

print(get_conversion_factor.invoke({"base_currency": "USD", "target_currency": "INR"}))

print(convert.invoke({"base_currency_value": 10, "conversion_rate": 85.16}))

# tool binding
# If using ChatOpenAI
# llm = ChatOpenAI()

# If using GoogleGenerativeAIEmbeddings
# Initialize ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI()

llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

messages = [
    HumanMessage(
        "What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd"
    )
]

# print(messages)

ai_message = llm_with_tools.invoke(messages)

messages.append(ai_message)

ai_message.tool_calls

import json

for tool_call in ai_message.tool_calls:
    # execute the 1st tool and get the value of conversion rate
    if tool_call["name"] == "get_conversion_factor":
        tool_message1 = get_conversion_factor.invoke(tool_call)
        # fetch this conversion rate
        conversion_rate = json.loads(tool_message1.content)["conversion_rate"]
        # append this tool message to messages list
        messages.append(tool_message1)
    # execute the 2nd tool using the conversion rate from tool 1
    if tool_call["name"] == "convert":
        # fetch the current arg
        tool_call["args"]["conversion_rate"] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)

# print(messages)

llm_with_tools.invoke(messages).content


# Initialize the Agent
agent_executor = initialize_agent(
    tools=[get_conversion_factor, convert],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,  # using ReAct pattern
    verbose=True,  # shows internal thinking
)

# Run the Agent
user_query = "Hi how are you?"

response = agent_executor.invoke({"input": user_query})

print(response)

```
                    User
                      │
                      │ "Convert 80 USD to INR"
                      ↓
                     LLM
                      │
                      │ Tool Call #1
                      ↓
          get_conversion_factor
                      │
                      │ Exchange-rate API
                      ↓
                Conversion Rate
                      │
                      ↓
                     LLM
                      │
                      │ Tool Call #2
                      ↓
                   convert
                      │
                      │ amount × rate
                      ↓
                Converted Amount
                      │
                      ↓
                     LLM
                      │
                      ↓
               Final Response
```